# Domain 1 – Tyre Degradation: XGBoost Modelling

This notebook covers:
1. Feature importance analysis
2. Train/test split visualisation
3. Model predictions vs actuals scatter plot
4. Quantile fan chart (Q10/Q50/Q90) for a sample stint


In [ ]:
import sys; sys.path.insert(0, '..')
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

from src.utils.paths import DOMAIN1_SILVER, DOMAIN1_MODELS
from src.models.train_degradation_model import (
    load_features, prepare_train_test_split, FEATURE_COLS
)
print('Imports OK')

## 1. Load Features and Split

In [ ]:
features_df = load_features()
X_train, y_train, X_test, y_test = prepare_train_test_split(features_df)

print(f'Train: {len(X_train):,} samples')
print(f'Test:  {len(X_test):,} samples')
print(f'Features: {list(X_train.columns)}')

## 2. Train/Test Split Visualisation

In [ ]:
# Show which rounds are train vs test
train_rounds = features_df.loc[X_train.index, 'round_number'] if 'round_number' in features_df.columns else None
test_rounds  = features_df.loc[X_test.index,  'round_number'] if 'round_number' in features_df.columns else None

if train_rounds is not None:
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.scatter(train_rounds, y_train, alpha=0.3, s=10, label='Train', color='steelblue')
    ax.scatter(test_rounds,  y_test,  alpha=0.5, s=10, label='Test',  color='tomato')
    ax.set_title('Train/Test Split: Degradation Rate vs Round Number')
    ax.set_xlabel('Round Number')
    ax.set_ylabel('Degradation Rate (s/lap)')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 3. Load Trained Models and Feature Importance

In [ ]:
model_main = XGBRegressor()
model_main.load_model(str(DOMAIN1_MODELS / 'degradation_main.json'))

# Feature importance
importances = pd.Series(
    model_main.feature_importances_,
    index=X_train.columns
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
importances.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('XGBoost Feature Importance (main model)')
ax.set_xlabel('Feature Importance (gain)')
plt.tight_layout()
plt.show()

## 4. Predictions vs Actuals

In [ ]:
y_pred = model_main.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter: actual vs predicted
axes[0].scatter(y_test, y_pred, alpha=0.4, s=20, color='steelblue')
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_title(f'Actual vs Predicted Degradation Rate\nMAE = {mae:.4f} s/lap')
axes[0].set_xlabel('Actual Degradation Rate (s/lap)')
axes[0].set_ylabel('Predicted Degradation Rate (s/lap)')
axes[0].legend()

# Residuals
residuals = y_pred - y_test.to_numpy()
axes[1].hist(residuals, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Residual (s/lap)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f'MAE:  {mae:.4f} s/lap')

## 5. Quantile Fan Chart (Q10 / Q50 / Q90)

In [ ]:
model_q10 = XGBRegressor(); model_q10.load_model(str(DOMAIN1_MODELS / 'degradation_q10.json'))
model_q50 = XGBRegressor(); model_q50.load_model(str(DOMAIN1_MODELS / 'degradation_q50.json'))
model_q90 = XGBRegressor(); model_q90.load_model(str(DOMAIN1_MODELS / 'degradation_q90.json'))

# Generate a sample stint trajectory (synthetic input grid)
# Vary stint_length while holding other features at median
sample = X_test.median().to_frame().T
stint_range = np.arange(5, 35)
rows = []
for sl in stint_range:
    r = sample.copy()
    if 'stint_length' in r.columns:
        r['stint_length'] = sl
    rows.append(r)
X_sample = pd.concat(rows, ignore_index=True)

pred_q10 = model_q10.predict(X_sample)
pred_q50 = model_q50.predict(X_sample)
pred_q90 = model_q90.predict(X_sample)

fig, ax = plt.subplots(figsize=(12, 6))
ax.fill_between(stint_range, pred_q10, pred_q90,
                alpha=0.25, color='steelblue', label='Q10–Q90 interval')
ax.plot(stint_range, pred_q50, color='steelblue', linewidth=2.5, label='Q50 (median prediction)')
ax.plot(stint_range, pred_q10, color='steelblue', linewidth=1, linestyle='--', label='Q10')
ax.plot(stint_range, pred_q90, color='steelblue', linewidth=1, linestyle='--', label='Q90')
ax.set_title('Quantile Fan Chart: Predicted Degradation Rate vs Stint Length')
ax.set_xlabel('Stint Length (laps)')
ax.set_ylabel('Predicted Degradation Rate (s/lap)')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Model Metrics Summary

In [ ]:
metrics_path = DOMAIN1_MODELS / 'model_metrics.json'
if metrics_path.exists():
    with open(metrics_path) as f:
        metrics = json.load(f)
    metrics_df = pd.DataFrame(metrics).T
    print('Model Performance Metrics:')
    print(metrics_df.round(4))
else:
    print('model_metrics.json not found. Run train_degradation_model.py first.')